In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

# ---------------- LOAD ----------------
data = pd.read_csv("dataset/catalog.csv")

# ---------------- CLEAN ----------------
data = data.dropna(subset=[
    "latitude", "longitude", "distance",
    "population", "landslide_size", "trigger", "fatalities"
])

# ---------------- TARGET ----------------
data["target"] = data["fatalities"].apply(lambda x: 1 if x > 0 else 0)

# ---------------- NORMALIZE TEXT ----------------
data["trigger"] = data["trigger"].str.lower()
data["landslide_size"] = data["landslide_size"].str.lower()

# Reduce trigger categories (VERY IMPORTANT 🔥)
def simplify_trigger(x):
    if "rain" in x or "downpour" in x:
        return "rain"
    elif "earthquake" in x:
        return "earthquake"
    elif "storm" in x or "cyclone" in x:
        return "storm"
    else:
        return "other"

data["trigger"] = data["trigger"].apply(simplify_trigger)

# ---------------- ONE HOT ENCODING ----------------
data = pd.get_dummies(data, columns=["landslide_size", "trigger"])

# ---------------- FEATURES ----------------
features = [col for col in data.columns if col.startswith((
    "latitude", "longitude", "distance", "population",
    "landslide_size_", "trigger_"
))]

X = data[features]
y = data["target"]

# ---------------- SPLIT ----------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------- SCALE ----------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------- MODEL ----------------
model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
model.fit(X_train_scaled, y_train)

# ---------------- EVALUATION ----------------
accuracy = model.score(X_test_scaled, y_test)
print("✅ Accuracy:", accuracy)

# ---------------- SAVE ----------------
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/landslide.pkl")
joblib.dump(scaler, "models/landslide_scaler.pkl")

print("✅ Model + scaler + columns saved")

✅ Accuracy: 0.8719723183391004
✅ Model + scaler + columns saved


In [15]:
# Example single input (same format as training BEFORE encoding)
sample_input = pd.DataFrame([{
    "latitude": 30.3165,
    "longitude": 78.0322,
    "distance": 4.5,
    "population": 12000,
    "landslide_size": "medium",
    "trigger": "rain"
}])

In [17]:
# normalize text
sample_input["trigger"] = sample_input["trigger"].str.lower()
sample_input["landslide_size"] = sample_input["landslide_size"].str.lower()

# same trigger simplification
def simplify_trigger(x):
    if "rain" in x or "downpour" in x:
        return "rain"
    elif "earthquake" in x:
        return "earthquake"
    elif "storm" in x or "cyclone" in x:
        return "storm"
    else:
        return "other"

sample_input["trigger"] = sample_input["trigger"].apply(simplify_trigger)

# one-hot encoding
sample_input = pd.get_dummies(sample_input)

In [23]:
import pandas as pd
import joblib

# load
model = joblib.load("ml-api/models/landslide.pkl")
scaler = joblib.load("ml-api/models/landslide_scaler.pkl")

# get expected columns from scaler
expected_columns = scaler.feature_names_in_

# sample input
sample_input = pd.DataFrame([{
    "latitude": 30.3165,
    "longitude": 78.0322,
    "distance": 4.5,
    "population": 12000,
    "landslide_size": "medium",
    "trigger": "rain"
}])

# preprocess
sample_input["trigger"] = sample_input["trigger"].str.lower()
sample_input["landslide_size"] = sample_input["landslide_size"].str.lower()

def simplify_trigger(x):
    if "rain" in x or "downpour" in x:
        return "rain"
    elif "earthquake" in x:
        return "earthquake"
    elif "storm" in x or "cyclone" in x:
        return "storm"
    else:
        return "other"

sample_input["trigger"] = sample_input["trigger"].apply(simplify_trigger)

# encoding
sample_input = pd.get_dummies(sample_input)

# 🔥 FIX: match scaler columns
for col in expected_columns:
    if col not in sample_input:
        sample_input[col] = 0

sample_input = sample_input[expected_columns]

# scale
sample_scaled = scaler.transform(sample_input)

# predict
prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0][1]

print("Prediction:", prediction)
print("Probability:", probability)

Prediction: 0
Probability: 0.0919261791973107
